# G1M1: Baseline (Urc1) vs Urc1_c3 (Unified Comparator)


## !!!Note: 
The comparison between the baseline method and the C3 method in this notebook is not actually accurate. (Because Ohref in the baseline method cannot be applied to the C3 method, so these two methods cannot actually be compared now.) Therefore, the scripts in this notebook are only examples for future work.



This notebook aligns Urc1_c3 with the same comparison workflow used by other Urc1-family models.

Compared models:
- Baseline: `Urc1` (fixed sliding intervals, `ln(h_since_last_start)`)
- C3 model: `Urc1_c3` (voltage-drop intervals, `ln(h_since_u_drop)`)

Main goals:
- Keep C3 core logic unchanged.
- Use comparator-friendly interfaces and aliases for both models.
- Run reference-specific comparison with `UnifiedModelComparator` (including optional GT metrics).

In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

from degradation_toolbox.Urc.Urc1 import Urc1
from degradation_toolbox.Urc.Urc1_c3 import Urc1_c3
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess

In [4]:
# =============================================================================
# CONFIGURATION SECTION - EDIT THESE TO CUSTOMIZE YOUR ANALYSIS
# =============================================================================
# Dataset and directories
DATASET_PATH = r"..\\..\\explore_data\\G1M1_new.parquet"
PREPROCESS_OUTPUT_DIR = r"..\\..\\explore_data\\output"
PLOTS_OUTPUT_DIR = r"..\\plots\\c3\\G1M1_comparison"
SAVE_PLOTS = True

# Reference condition configurations: Low, Medium, High
REF_CONFIGS = {
    "Low": {
        "Iref": 0.3,
        "Tref": 58,
        "OHref": 11,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 100,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.48,
        "Tref": 57,
        "OHref": 100,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_regression_full_coverage.csv",
    },
}

# Model fitting common config (shared across baseline and c3)
COMMON_CONFIG = {
    "Iref": [cfg["Iref"] for cfg in REF_CONFIGS.values()],
    "Tref": 60,
    "OHref": 72,
    "ref_config": REF_CONFIGS,
    "threshold": 1e6,
    "i_off": 0.1,
    "u_off": 1.3,
    "plot_fit": 0,
    "data_filter_i_min": 0.1,
    "data_filter_U_min": 1.4,
    "data_filter_U_max": 2.3,
    "data_filter_T_min": 50,
    "data_filter_T_max": 65,
    "data_filter_h_since_last_start_min": 0.5,
}

# Baseline-only config
BASELINE_CONFIG = {
    "len_interval": 2,
    "slide": 1,
    "min_num_data_required_for_fit": 300,
}

# c3-only config
C3_EXTRA_CONFIG = {
    "u_drop_threshold": 1.7,
    "min_num_data_required_for_fit": 300,
    "data_filter_h_since_u_drop_min": 0.01,
}

# Comparison settings
SHOW_GT_METRICS = True
SHOW_ALL_COND_METRICS = True
REF_ORDER = list(REF_CONFIGS.keys())

In [5]:
# =============================================================================
# 1. DATA LOADING & PREPROCESSING
# =============================================================================
print("=" * 80)
print("STEP 1: Data Loading & Preprocessing")
print("=" * 80)

preprocessor = GMpreprocess(file_path=DATASET_PATH, output_dir=PREPROCESS_OUTPUT_DIR)
data = preprocessor.run()
dataset_name = preprocessor.name

print(f"Dataset: {dataset_name}, shape: {data.shape}")
print(f"Time range: {data.index.min()} -> {data.index.max()}\n")

# Shared preprocess for baseline only (c3 uses its own dynamic preprocess)
shared_pre = Urc1.preprocess_once(
    data,
    i_off=COMMON_CONFIG["i_off"],
    u_off=COMMON_CONFIG["u_off"],
    data_filter_i_min=COMMON_CONFIG["data_filter_i_min"],
    data_filter_U_min=COMMON_CONFIG["data_filter_U_min"],
    data_filter_U_max=COMMON_CONFIG["data_filter_U_max"],
    data_filter_T_min=COMMON_CONFIG["data_filter_T_min"],
    data_filter_T_max=COMMON_CONFIG["data_filter_T_max"],
    data_filter_h_since_last_start_min=COMMON_CONFIG["data_filter_h_since_last_start_min"],
)
print(f"Shared preprocessed rows (baseline): {len(shared_pre)}\n")

STEP 1: Data Loading & Preprocessing
=== 1. Loading & Preprocessing: G1M1_new ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_1' -> ID: '1'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   ..\\..\\explore_data\\output\G1M1_new_20260501_210252.parquet

=== GMpreprocess Pipeline Completed Successfully ===
Dataset: G1M1_new, shape: (2529217, 3)
Time range: 2021-02-12 00:00:00 -> 2025-12-09 09:59:00

Shared preprocessed rows (baseline): 1069062



In [6]:
# =============================================================================
# 2. TRAIN MODELS
# =============================================================================
print("=" * 80)
print("STEP 2: Model Training")
print("=" * 80)

models = {}

print("\n  -> Training Baseline (Urc1) model...")
urc_baseline = Urc1(
    data=data,
    name=dataset_name,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
    **BASELINE_CONFIG,
)
models["Baseline"] = urc_baseline
print("  ✓ Baseline training completed")

print("\n  -> Training Urc1_c3 model...")
urc_c3 = Urc1_c3(
    data=data,
    name=dataset_name,
    **COMMON_CONFIG,
    **C3_EXTRA_CONFIG,
)
models["Urc1_c3"] = urc_c3
print("  ✓ Urc1_c3 training completed")

print(f"\n✓ {len(models)} models trained successfully\n")

STEP 2: Model Training

  -> Training Baseline (Urc1) model...
  ✓ Baseline training completed

  -> Training Urc1_c3 model...
Data preprocessing ...
Detected 23368 intervals (voltage dropping below 1.7V).
Before preprocess: 2529217 -> After: 1053733 points.
  (Filters: i>0.1 A/cm2, 1.4 V<U<2.3 V, 50 C<T<65 C, h_since_u_drop>0.01 h)
Voltage model fitting ...
Fitting Stats: 22854 intervals low data, 0 fit failed.
233 out of 23368 fitting results are reliable.
  ✓ Urc1_c3 training completed

✓ 2 models trained successfully



In [7]:
# =============================================================================
# 3. C3 INTERVAL DIAGNOSTICS
# =============================================================================
print("=" * 80)
print("STEP 3: Urc1_c3 Interval Diagnostics")
print("=" * 80)

interval_summary = urc_c3.get_intervals_summary()
print(f"Total intervals: {len(interval_summary)}")
print(f"Fit Success intervals: {(interval_summary['status'] == 'Fit Success').sum()}")
print(
    "Insufficient Data intervals:",
    interval_summary["status"].astype(str).str.contains("Insufficient", na=False).sum(),
)
print("\nStatus counts:")
print(interval_summary["status"].value_counts(dropna=False).head(10))

display(interval_summary.head(20))

STEP 3: Urc1_c3 Interval Diagnostics
Total intervals: 23368
Fit Success intervals: 260
Insufficient Data intervals: 22854

Status counts:
status
Insufficient Data (0 < 300)     1231
Insufficient Data (14 < 300)    1049
Insufficient Data (13 < 300)     832
Insufficient Data (4 < 300)      773
Insufficient Data (11 < 300)     760
Insufficient Data (9 < 300)      757
Insufficient Data (3 < 300)      754
Insufficient Data (2 < 300)      737
Insufficient Data (12 < 300)     734
Insufficient Data (15 < 300)     715
Name: count, dtype: int64


,start_time,end_time,interval_h,n_valid_points,status,quality,R2
0,2021-02-12 00:00:00,2021-02-12 12:17:00,12.283333,18,Insufficient Data (18 < 300),failed,NaN
1,2021-02-12 12:17:00,2021-02-12 13:51:00,1.566667,47,Insufficient Data (47 < 300),failed,NaN
2,2021-02-12 13:51:00,2021-02-20 06:02:00,184.183333,10489,Fit Success,Vertex of UI curve is between 0 and 2.5A/cm2.,0.996367
3,2021-02-20 06:02:00,2021-02-25 15:35:00,129.550000,7770,High Cond (2.8e+08),good,0.894799
4,2021-02-25 15:35:00,2021-03-03 00:52:00,129.283333,7753,High Cond (8.6e+07),good,0.832309
5,2021-03-03 00:52:00,2021-03-03 20:35:00,19.716667,1180,High Cond (1.7e+08),good,0.942195
6,2021-03-03 20:35:00,2021-04-02 14:01:00,713.433333,42805,High Cond (2.2e+06),good,0.988359
7,2021-04-02 14:01:00,2021-04-02 22:01:00,8.000000,396,Fit Success,good,0.991687
8,2021-04-02 22:01:00,2021-04-02 23:08:00,1.116667,66,Insufficient Data (66 < 300),failed,NaN
9,2021-04-02 23:08:00,2021-04-02 23:12:00,0.066667,3,Insufficient Data (3 < 300),failed,NaN


In [8]:
# =============================================================================
# 4. COMPARATOR SETUP & GT LOADING
# =============================================================================
print("\n" + "=" * 80)
print("STEP 4: Initialize Comparator & Load Ground Truth")
print("=" * 80)

comparator = UnifiedModelComparator(models)
print(f"✓ UnifiedModelComparator initialized with {len(models)} models\n")

gt_loaded_count = 0
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    gt_file = ref_cfg["gt_file"]
    gt_path = Path(gt_file)
    if not gt_path.is_absolute():
        gt_path = Path.cwd() / gt_path
    gt_path = gt_path.resolve()
    print(f"Looking for GT file: {gt_path}")

    if gt_path.exists():
        try:
            gt_data = pd.read_csv(gt_path, index_col=0, parse_dates=True)
            colmap = {str(c).strip().lower(): c for c in gt_data.columns}
            candidate_cols = ["gt_uref_regression", "voltage", "uref", "gt_uref"]
            selected_col = None
            for c in candidate_cols:
                if c in colmap:
                    selected_col = colmap[c]
                    break

            if selected_col is not None:
                gt_series = gt_data[selected_col]
            elif len(gt_data.columns) == 1:
                gt_series = gt_data.iloc[:, 0]
                selected_col = gt_data.columns[0]
            else:
                raise ValueError(
                    f"Cannot identify voltage column in {gt_path}. Columns: {gt_data.columns.tolist()}"
                )

            gt_series = gt_series.dropna()
            comparator.set_ground_truth(gt_series, iref=iref)
            gt_loaded_count += 1
            print(
                f"  ✓ Loaded GT for {ref_name} (Iref={iref}): {len(gt_series)} points [column: {selected_col}]"
            )
        except Exception as e:
            print(f"  ✗ Failed to load GT for {ref_name}: {e}")
    else:
        print(f"  ⚠ GT file NOT found: {gt_path}")

has_gt = gt_loaded_count > 0 and SHOW_GT_METRICS
print(f"\n{'=' * 80}")
print(f"GT status: {gt_loaded_count}/{len(REF_CONFIGS)} reference conditions loaded")
print(f"GT metrics will be {'ENABLED ✓' if has_gt else 'DISABLED'}")
print(f"{'=' * 80}\n")


STEP 4: Initialize Comparator & Load Ground Truth
✓ UnifiedModelComparator initialized with 2 models

Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_regression_full_coverage.csv
  ✓ Loaded GT for Low (Iref=0.3): 1762 points [column: gt_uref_regression]
Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_regression_full_coverage.csv
  ✓ Loaded GT for Medium (Iref=1.0): 1762 points [column: gt_uref_regression]
Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_regression_full_coverage.csv
  ✓ Loaded GT for High (Iref=1.48): 1384 points [column: gt_uref_regression]

GT stat

In [9]:
# =============================================================================
# 5. REFERENCE-SPECIFIC ANALYSIS (UNIFIED COMPARATOR)
# =============================================================================
print("=" * 80)
print("STEP 5: Reference-Specific Metrics & Comparison")
print("=" * 80)

rate_tables = []
for ref_name in REF_ORDER:
    ref_cfg = REF_CONFIGS[ref_name]
    iref = ref_cfg["Iref"]
    tref = ref_cfg["Tref"]
    ohref = ref_cfg["OHref"]

    print(f"\n▓▓▓ REFERENCE CONDITION: {ref_name} (Iref={iref}, Tref={tref}, OHref={ohref}) ▓▓▓")

    df_metrics = comparator.compare_all(
        i_target=iref,
        outlier_threshold_method="2rmse",
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
        include_gt_metrics=has_gt,
    )
    display(Markdown(df_metrics.to_markdown(index=False)))

    rate_tables.append(
        df_metrics[["Model Name", "Target Current (A/cm2)", "Degradation Rate (uV/h)", "Slope Sigma (uV/h)"]].assign(
            Reference=ref_name
        )
    )

    try:
        comparator.plot_interactive_trends(
            target_i=iref,
            show_gt=has_gt,
            save=SAVE_PLOTS,
            output_dir=PLOTS_OUTPUT_DIR,
            uncertainty_style="band",
            uncertainty_opacity=0.12,
            show_series_line=True,
            rate_precision=6,
        )
        print("✓ Trend plot finished")
    except Exception as e:
        print(f"⚠ Trend plot failed: {e}")

    comparator.print_comparison_report(
        i_target=iref,
        include_gt_metrics=has_gt,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
    )

STEP 5: Reference-Specific Metrics & Comparison

▓▓▓ REFERENCE CONDITION: Low (Iref=0.3, Tref=58, OHref=11) ▓▓▓


| Model Name   |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:-------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline     |                      0.3 |             15.854 |              1098 |                   1.25211 |      18.885 |                    0 |               0.925 |              10 |          0.91 |             390.861 |          4.105 |               4.745 |           13.287 |                      1286 | all_fitted   |         19.114 |         5.783 |                 1098 |
| Urc1_c3      |                      0.3 |             32.578 |               233 |                   2.08613 |     103.785 |                    0 |               0.793 |               2 |          0.86 |            1514.78  |          3.865 |               5.965 |           12.98  |                       514 | all_fitted   |        104.992 |        19.978 |                  233 |

✓ Trend plot finished
  MODEL COMPARISON REPORT @ 0.3 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      1098
  Deg Rate:         1.252113 μV/h
  RMSE:             18.885 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.925
  Outliers:         10 (0.9%)
  Max Residual:     390.861 mV
  Mean SE:          4.105 mV
  Cond (Median):    10^4.7
  Cond Scope:       all_fitted (n=1286)
  GT RMSE:          19.114 mV (n=1098)
  GT MAE:           5.783 mV

📊 Urc1_c3
------------------------------------------------------------
  Data Points:      233
  Deg Rate:         2.086131 μV/h
  RMSE:             103.785 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.793
  Outliers:         2 (0.9%)
  Max Residual:     1514.780 mV
  Mean SE:          3.865 mV
  Cond (Median):    10^6.0
  Cond Scope:       all_fitted (n=514)
  GT RMSE:          104.992 mV (n=233)
  GT MAE:           19.978 mV

🏆 Best RMSE (vs model data):    Baseline
🏆 Best Mono

| Model Name   |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:-------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline     |                        1 |             15.854 |              1098 |                   5.20858 |      18.873 |                    0 |               0.959 |              41 |          3.73 |             141.761 |          4.137 |               4.745 |           13.287 |                      1286 | all_fitted   |         20.604 |        15.377 |                 1098 |
| Urc1_c3      |                        1 |             32.578 |               233 |                   5.13951 |      30.25  |                    0 |               0.836 |               8 |          3.43 |             151.299 |          3.404 |               5.965 |           12.98  |                       514 | all_fitted   |         32.772 |        21.202 |                  233 |

✓ Trend plot finished
  MODEL COMPARISON REPORT @ 1.0 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      1098
  Deg Rate:         5.208581 μV/h
  RMSE:             18.873 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.959
  Outliers:         41 (3.7%)
  Max Residual:     141.761 mV
  Mean SE:          4.137 mV
  Cond (Median):    10^4.7
  Cond Scope:       all_fitted (n=1286)
  GT RMSE:          20.604 mV (n=1098)
  GT MAE:           15.377 mV

📊 Urc1_c3
------------------------------------------------------------
  Data Points:      233
  Deg Rate:         5.139507 μV/h
  RMSE:             30.250 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.836
  Outliers:         8 (3.4%)
  Max Residual:     151.299 mV
  Mean SE:          3.404 mV
  Cond (Median):    10^6.0
  Cond Scope:       all_fitted (n=514)
  GT RMSE:          32.772 mV (n=233)
  GT MAE:           21.202 mV

🏆 Best RMSE (vs model data):    Baseline
🏆 Best Monoto

| Model Name   |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:-------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline     |                     1.48 |             15.854 |              1098 |                   10.9523 |      55.143 |                    0 |               0.914 |              65 |          5.92 |             326.556 |          5.353 |               4.745 |           13.287 |                      1286 | all_fitted   |         64.473 |        48.601 |                  868 |
| Urc1_c3      |                     1.48 |             32.578 |               233 |                   10.351  |      69.913 |                    0 |               0.73  |              14 |          6.01 |             203.395 |          6.846 |               5.965 |           12.98  |                       514 | all_fitted   |         72.938 |        57.777 |                  213 |

✓ Trend plot finished
  MODEL COMPARISON REPORT @ 1.48 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      1098
  Deg Rate:         10.952308 μV/h
  RMSE:             55.143 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.914
  Outliers:         65 (5.9%)
  Max Residual:     326.556 mV
  Mean SE:          5.353 mV
  Cond (Median):    10^4.7
  Cond Scope:       all_fitted (n=1286)
  GT RMSE:          64.473 mV (n=868)
  GT MAE:           48.601 mV

📊 Urc1_c3
------------------------------------------------------------
  Data Points:      233
  Deg Rate:         10.351000 μV/h
  RMSE:             69.913 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.730
  Outliers:         14 (6.0%)
  Max Residual:     203.395 mV
  Mean SE:          6.846 mV
  Cond (Median):    10^6.0
  Cond Scope:       all_fitted (n=514)
  GT RMSE:          72.938 mV (n=213)
  GT MAE:           57.777 mV

🏆 Best RMSE (vs model data):    Baseline
🏆 Best Mon

In [10]:
# =============================================================================
# 6. CROSS-MODEL DIAGNOSTICS
# =============================================================================
print("\n" + "=" * 80)
print("STEP 6: Cross-Model Diagnostics")
print("=" * 80)

try:
    print("\n-> Plotting fit quality (RMSE & R² distributions)...")
    comparator.plot_fit_quality(save=SAVE_PLOTS)
    print("✓ Fit quality completed")
except Exception as e:
    print(f"⚠ Fit quality plot failed: {e}")

try:
    print("\n-> Plotting coefficient diagnostics...")
    comparator.plot_coefficient_diagnostic(save=SAVE_PLOTS, include_c6=False)
    print("✓ Coefficient diagnostic completed")
except Exception as e:
    print(f"⚠ Coefficient diagnostic failed: {e}")

try:
    print("\n-> Plotting coverage Gantt...")
    comparator.plot_coverage_gantt(save=SAVE_PLOTS)
    print("✓ Coverage Gantt completed")
except Exception as e:
    print(f"⚠ Coverage Gantt failed: {e}")


STEP 6: Cross-Model Diagnostics

-> Plotting fit quality (RMSE & R² distributions)...
✓ Fit quality completed

-> Plotting coefficient diagnostics...
✓ Coefficient diagnostic completed

-> Plotting coverage Gantt...
✓ Coverage Gantt completed


In [11]:
# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\n✓ Trained models: {len(models)}")
print(f"✓ Reference conditions analyzed: {len(REF_CONFIGS)}")
print(f"✓ Ground truth data: {'Loaded ✓' if has_gt else 'Not available'}")
print(f"✓ Output plots: {PLOTS_OUTPUT_DIR}/" if SAVE_PLOTS else "✓ Plots displayed (not saved)")
print("\nKey settings:")
print(f"  - All condition metrics: {SHOW_ALL_COND_METRICS}")
print(f"  - GT metrics: {SHOW_GT_METRICS}")
print(f"  - Save plots: {SAVE_PLOTS}")

if rate_tables:
    print("\nDegradation-rate summary across references:")
    display(pd.concat(rate_tables, ignore_index=True))


ANALYSIS COMPLETE

✓ Trained models: 2
✓ Reference conditions analyzed: 3
✓ Ground truth data: Loaded ✓
✓ Output plots: ..\\plots\\c3\\G1M1_comparison/

Key settings:
  - All condition metrics: True
  - GT metrics: True
  - Save plots: True

Degradation-rate summary across references:


,Model Name,Target Current (A/cm2),Degradation Rate (uV/h),Slope Sigma (uV/h),Reference
0,Baseline,0.30,1.252113,0.0,Low
1,Urc1_c3,0.30,2.086131,0.0,Low
2,Baseline,1.00,5.208581,0.0,Medium
3,Urc1_c3,1.00,5.139507,0.0,Medium
4,Baseline,1.48,10.952308,0.0,High
5,Urc1_c3,1.48,10.351000,0.0,High
